# E1.1 · Why point-in-time control testing fails for AI

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.0 · Start here — what AI governance means](https://spbreed.github.io/cyber-commons/lessons/E1.0.html)**.

| | |
|---|---|
| Tools used | promptfoo |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

You tested the control in March and signed the assertion. The prompt changed in April, the model in May, and the tool scope in June. The assertion is still on file and has not described anything real since the day it was written.

> **At CyberTravels.** The control test that passed in March described a CyberTravels with no payments scope, no repository access and no vector store. Nothing about it was wrong; everything about it is stale.

## 2 · The framework

```
   march      test the control, sign the assertion
   april      the prompt changes
   may        the model version changes
   june       the tool scope changes
   december   the assertion is still on file

   point-in-time assurance for a system that changes between tests
   describes a system that no longer exists
```

Classical control testing has a simple shape: a control is designed, an auditor
tests it once or twice a year, and a passing test is recorded for the period.

That works when the thing being tested changes only through a process that
generates evidence. For an agent, the four things that change its behaviour are:

- the **model version** — changed by your provider, possibly without notice,
- the **prompt** — edited in a console,
- the **tool manifest** — a config change,
- the **approval settings** — a toggle in an admin UI.

None of them is a code change. None generates a change record. All of them
invalidate the conditions the control was tested under.

The honest consequence is that a control tested six months ago is not passing —
it is **unevidenced**, which is a third state most GRC tooling cannot represent.
Introducing that third state is the whole of this lesson.

## 3 · Demo — the same evidence, two ways of reading it

In [ ]:
import time
from dataclasses import dataclass, field

now = time.time(); DAY = 86400

@dataclass
class ControlTest:
    cid: str
    passed: bool
    evidence: str
    tested_at: float
    valid_for_days: float = 30

    def age_days(self, at): return (at - self.tested_at) / DAY
    def point_in_time(self, at): return "PASS" if self.passed else "FAIL"
    def continuous(self, at):
        if self.age_days(at) > self.valid_for_days: return "STALE"
        return "PASS" if self.passed else "FAIL"

TESTS = [
 ControlTest("AC-1", True,  "act chain sampled from gateway logs", now -   3*DAY),
 ControlTest("AC-2", True,  "delegation refusal regression suite", now -   9*DAY),
 ControlTest("SB-1", True,  "egress denial evidence",              now -  45*DAY),
 ControlTest("SB-2", True,  "approval gate screenshot",            now - 210*DAY),
 ControlTest("EV-1", True,  "audit sample of 50 agent actions",    now -   5*DAY),
 ControlTest("DR-1", False, "drift alerting not deployed",         now),
]
REQUIRED = ["AC-1", "AC-2", "SB-1", "SB-2", "EV-1", "DR-1", "EV-2", "ST-1"]

print(f"{'control':9s}{'age (days)':>12}{'point-in-time':>16}{'continuous':>13}")
print("-" * 52)
by_id = {t.cid: t for t in TESTS}
for cid in REQUIRED:
    t = by_id.get(cid)
    if t is None:
        print(f"{cid:9s}{'—':>12}{'(not tested)':>16}{'NO EVIDENCE':>13}")
        continue
    print(f"{cid:9s}{t.age_days(now):>12.0f}{t.point_in_time(now):>16}{t.continuous(now):>13}")

## 4 · Where it breaks — the two numbers those readings produce

In [ ]:
def posture(tests, required, at, mode):
    by_id = {t.cid: t for t in tests}
    passing = 0
    for cid in required:
        t = by_id.get(cid)
        if t is None: continue
        state = t.point_in_time(at) if mode == "point-in-time" else t.continuous(at)
        passing += state == "PASS"
    return passing, round(passing/len(required), 3)

for mode in ("point-in-time", "continuous"):
    n, pct = posture(TESTS, REQUIRED, now, mode)
    print(f"{mode:16s} {n}/{len(REQUIRED)} controls passing = {pct:.0%}")

print("\nThe difference is entirely SB-1 and SB-2, which nobody did anything")
print("wrong to. Time simply passed, and the agent they were tested against")
print("has had two model upgrades since.")

## 5 · The control — a freshness window per control, derived from drift

The window is not an audit-calendar choice. It comes from **how fast the thing the control tests actually changes.**

In [ ]:
DRIFT_RATE = {          # observed TVD/day for what each control depends on
 "AC-1": 0.0005,        # identity model changes slowly
 "AC-2": 0.0005,
 "SB-1": 0.0020,        # egress needs change with new integrations
 "SB-2": 0.0090,        # tool manifests change weekly
 "EV-1": 0.0010,
 "DR-1": 0.0090,
}
TOLERANCE = 0.25

def window(cid):
    r = DRIFT_RATE.get(cid)
    return int(TOLERANCE / r) if r else 90

print(f"{'control':9s}{'drift/day':>12}{'window (days)':>15}{'current age':>13}{'state':>9}")
print("-" * 60)
for cid in REQUIRED:
    t = by_id.get(cid)
    w = window(cid)
    if t is None:
        print(f"{cid:9s}{'—':>12}{w:>15}{'—':>13}{'NO EVIDENCE':>9}")
        continue
    t.valid_for_days = w
    print(f"{cid:9s}{DRIFT_RATE.get(cid, 0):>12.4f}{w:>15}{t.age_days(now):>13.0f}"
          f"{t.continuous(now):>9}")

n, pct = posture(TESTS, REQUIRED, now, "continuous")
print(f"\nwith drift-derived windows: {n}/{len(REQUIRED)} = {pct:.0%} currently evidenced")
assert pct < 0.6
print("\nSB-2 tests a tool manifest that changes weekly; a 210-day-old screenshot")
print("cannot evidence it. Saying so is the control, not a criticism of anyone.")

## What you just proved

Point-in-time reading reports 5 of 8 controls passing (63%); the continuous reading reports 3 of 8 (38%), with SB-1 and SB-2 STALE and EV-2 and ST-1 having no evidence at all. Drift-derived windows tighten SB-2 to roughly 27 days, confirming a 210-day-old screenshot cannot evidence a weekly-changing manifest.

## Your turn

Pick your three most important AI controls and set a freshness window for each from the observed change rate of what it tests. Then recompute your posture. The number will drop, and it will be the first honest one you have had.

---

**Next → [E1.2 · Building the AI and agent inventory](https://spbreed.github.io/cyber-commons/lessons/E1.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*